# autofollowdown — full backend flow demo (real results on a GPU)

This notebook runs the **whole flow** and shows **real results from every backend** we integrate: `native` · `Microsoft NNI` · `NVIDIA ModelOpt` · `llm-compressor (vLLM)`.

Built for **Google Colab + T4 GPU** (free tier): **Runtime → Change runtime type → T4 GPU**, then *Runtime → Run all*. A T4 supports INT8, so all backends run here.

**The flow:**  profile the model → `recommend` the best library (and why) → run each backend via the same `compress_with(model, backend)` call → compare the real results side by side.

Repo: https://github.com/peetwan/autofollowdown

## 0. Install autofollowdown + every backend

In [ ]:
!pip install -q "git+https://github.com/peetwan/autofollowdown#egg=autofollowdown[examples]"
!pip install -q nni "nvidia-modelopt[torch]" llmcompressor

In [ ]:
import torch
from autofollowdown import all_backends
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('GPU:', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU only')
print()
for b in all_backends():
    print(f'{b.name:40} {"installed ✓" if b.is_available() else "missing"}')

## 1. Profile the model and let autofollowdown recommend a library (the *why*)

We use a small but real LLM — `Qwen/Qwen3-0.6B`. `explain()` profiles it and ranks the libraries with the reasoning behind each.

In [ ]:
from autofollowdown import profile_from_pretrained, recommend_profile
LLM_ID = 'Qwen/Qwen3-0.6B'      # try Qwen2.5-1.5B / 3B on a bigger GPU
prof = profile_from_pretrained(LLM_ID)   # reads the config only — no weight download
_, recs = recommend_profile(prof)
print(f'{LLM_ID}: family={prof.family}, ~{prof.num_params/1e6:.0f}M params')
for r in sorted(recs, key=lambda r: r.score, reverse=True):
    print(f'  [{r.score:.2f}] {r.backend:34} {r.scheme:18} — {r.rationale[:60]}')

## 2. LLM library shoot-out — same model, same metric (WikiText-2 perplexity)

We compress `Qwen3-0.6B` with each library through the unified `compress_with(model, backend)` call and score every result on WikiText-2 perplexity (lower = better) plus on-disk size. Each step is independent and skips cleanly if a backend isn't available.

In [ ]:
import copy, os, tempfile, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from autofollowdown import (ModelCompressor, compress_with,
                            evaluate_perplexity, load_wikitext2, model_disk_size_mb)

tok = AutoTokenizer.from_pretrained(LLM_ID)
text = load_wikitext2('test')[:3000]
results = []

def ppl(model):
    return evaluate_perplexity(model, tok, text, stride=512, max_length=1024, device=DEVICE)

def hf_dir_mb(model):
    d = tempfile.mkdtemp(); model.save_pretrained(d)
    return sum(os.path.getsize(os.path.join(d, f)) for f in os.listdir(d)) / 1e6

# --- baseline (fp16/fp32) ---
dtype = torch.float16 if DEVICE == 'cuda' else torch.float32
base = AutoModelForCausalLM.from_pretrained(LLM_ID, torch_dtype=dtype).to(DEVICE).eval()
results.append(('baseline', f'{dtype}'.split('.')[-1], hf_dir_mb(base), ppl(base)))
print('baseline:', results[-1])

In [ ]:
# --- native: portable INT8 dynamic (CPU) ---
try:
    m = ModelCompressor(copy.deepcopy(base).cpu()).quantize(method='int8', approach='dynamic').model
    results.append(('native', 'int8-dynamic', model_disk_size_mb(m),
                    evaluate_perplexity(m, tok, text, stride=512, max_length=1024, device='cpu')))
    print('native:', results[-1]); del m
except Exception as e:
    print('native skipped:', e)

In [ ]:
# --- llm-compressor: 4-bit GPTQ (W4A16) ---
try:
    from llmcompressor.modifiers.gptq import GPTQModifier
    m = AutoModelForCausalLM.from_pretrained(LLM_ID, torch_dtype='auto', device_map=DEVICE)
    compress_with(m, 'llmcompressor',
                  recipe=GPTQModifier(targets='Linear', scheme='W4A16', ignore=['lm_head']),
                  dataset='open_platypus', num_calibration_samples=128)
    results.append(('llm-compressor', 'GPTQ W4A16', hf_dir_mb(m), ppl(m)))
    print('llm-compressor:', results[-1]); del m
except Exception as e:
    print('llm-compressor skipped:', e)

In [ ]:
# --- NVIDIA ModelOpt: INT8 SmoothQuant PTQ ---
try:
    import modelopt.torch.quantization as mtq
    m = AutoModelForCausalLM.from_pretrained(LLM_ID, torch_dtype=torch.float16).to(DEVICE)
    calib = [tok(t, return_tensors='pt').input_ids.to(DEVICE) for t in
             ['Compression makes models smaller.', 'Quantization lowers precision.',
              'Knowledge distillation trains a student.']]
    compress_with(m, 'modelopt', calibration_data=calib, quant_cfg=mtq.INT8_SMOOTHQUANT_CFG)
    results.append(('NVIDIA ModelOpt', 'INT8 SmoothQuant', None, ppl(m)))
    print('modelopt:', results[-1]); del m
except Exception as e:
    print('modelopt skipped:', e)

### Side-by-side results

In [ ]:
from IPython.display import Markdown, display
b_size, b_ppl = results[0][2], results[0][3]
rows = '| Library | Method | Size (MB) | Perplexity↓ | Size× | ΔPPL |\n|---|---|---|---|---|---|\n'
for name, method, size, p in results:
    sx = f'{b_size/size:.2f}×' if (size and name != 'baseline') else '—'
    dp = f'{p-b_ppl:+.2f}' if name != 'baseline' else '—'
    rows += f'| {name} | {method} | {size:.0f} | {p:.2f} | {sx} | {dp} |\n' if size else \
            f'| {name} | {method} | — | {p:.2f} | — | {dp} |\n'
display(Markdown(rows))

## 3. Vision track — native vs NNI (structured pruning is NNI's specialty)

NNI targets CNNs: its `ModelSpeedup` physically removes pruned channels (real FLOP/size reduction), so we compare it to the native engine on a small CNN.

In [ ]:
import torch.nn as nn
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64,10))
    def forward(self,x): return self.net(x)

def nparams(m): return sum(p.numel() for p in m.parameters())
vis = []
base_cnn = CNN(); vis.append(('baseline', nparams(base_cnn)))

# native: magnitude prune + INT8
m = ModelCompressor(copy.deepcopy(base_cnn)).prune(0.5).quantize(approach='dynamic').model
vis.append(('native prune+int8', nparams(m)))

# NNI: structured filter pruning + ModelSpeedup (real channel removal)
try:
    m = copy.deepcopy(base_cnn)
    compress_with(m, 'nni', dummy_input=torch.randn(1,3,32,32), sparsity=0.5)
    vis.append(('NNI structured + speedup', nparams(m)))
except Exception as e:
    print('NNI skipped:', e)

rows = '| Method | Params | vs baseline |\n|---|---|---|\n'
b = vis[0][1]
for name, n in vis:
    rows += f'| {name} | {n:,} | {b/max(n,1):.2f}× fewer |\n'
display(Markdown(rows))

## Summary

You just ran the whole flow and saw **real results from all four backends** through one consistent API:

- **native** — portable INT8 (and pruning/distillation), no extra deps;
- **llm-compressor** — 4-bit GPTQ on the LLM, vLLM-ready;
- **NVIDIA ModelOpt** — INT8 SmoothQuant PTQ on the LLM (TensorRT-exportable);
- **Microsoft NNI** — structured pruning that physically shrinks the CNN.

autofollowdown picked the best library for the model (`recommend`), ran each one with the same `compress_with(model, backend)` call, and benchmarked them on the same metric — so the comparison is honest and reproducible. Swap `LLM_ID` for a bigger Qwen to push further.